In [0]:
""" 
Weekly AI briefing
Pulls latest week's numbers from gold, sends to databricks claude endpoint, 
writes the result to gold.weekly_briefing. Runs as job task 3, after the
pipeline update task.

"""

import time
import json
from datetime import date
 
import mlflow.deployments
from pyspark.sql import functions as F

# Configuration

CATALOG = "fed_monitor"
ENDPOINT = "databricks-gpt-oss-120b"
 

In [0]:
# Pull and clean most recent row from gold tables, fed, bank, and rates

fed_row = spark.sql(f"""
    SELECT * FROM {CATALOG}.gold.fed_balance_sheet_weekly
    ORDER BY observation_date DESC LIMIT 1
""").collect()
 
bank_row = spark.sql(f"""
    SELECT * FROM {CATALOG}.gold.bank_balance_sheet_weekly
    ORDER BY observation_date DESC LIMIT 1
""").collect()
 
rates_row = spark.sql(f"""
    SELECT * FROM {CATALOG}.gold.rates_daily
    WHERE sofr IS NOT NULL
    ORDER BY observation_date DESC LIMIT 1
""").collect()
 
if not fed_row or not bank_row or not rates_row:
    raise ValueError("One or more gold tables returned no rows — aborting briefing.")
 
fed_row, bank_row, rates_row = fed_row[0].asDict(), bank_row[0].asDict(), rates_row[0].asDict()
 
as_of: date = fed_row["observation_date"]

def _clean(d):
    return {k: (v.isoformat() if isinstance(v, date) else v) for k, v in d.items()}

data = {
    "fed_balance_sheet": _clean(fed_row),
    "bank_balance_sheet": _clean(bank_row),
    "rates": _clean(rates_row),
}

In [0]:
# Build prompt and mlflow call

prompt = f"""
You are a macroeconomic analyst writing a weekly briefing for an audience that isn't a macroeconomics expert. He is a 70 year old veteran retail investor, mostly in stocks (currently largely in tech (BB), energy (some mid-large sized Canadian oil/energy companies), and other diverse holdings). So he has lots of knowledge of investments and economics but not lots of macroeconomics expertise (i.e. he doesn't watch these metrics regularly). So he is unfamiliar with the details of Federal Reserve liquidity, bank balance sheets, and money-market plumbing.

Analyze the supplied data in exactly seven sentences:

1. Describe the most important current level and trend in the Federal Reserve
   balance-sheet data, using concrete numbers.
2. Explain in plain English whether those Fed developments appear to be adding
   liquidity, withdrawing liquidity, or having little net effect, and why.
3. Describe the most important current level and trend in the commercial-bank
   balance-sheet data, using concrete numbers.
4. Explain what the bank data suggests about credit growth, deposit funding,
   liquidity, lending appetite, or financial leverage.
5. Describe the most important development in short-term interest rates and
   money-market spreads, using concrete numbers.
6. Explain whether the rate relationships indicate orderly funding conditions,
   tightening liquidity, or signs of money-market stress.
7. Give a balanced overall assessment of what the combined signals imply for
   the near-term equity-market environment, without making a definitive market
   forecast or recommending that the reader buy or sell anything.

Analytical rules:
- Focus on trends and changes rather than merely listing current balances.
- Compare week-over-week and year-over-year figures when both are available.
- Give greater weight to year-over-year changes when assessing persistent
  trends, but mention a weekly change when it materially alters current liquidity.
- Do not interpret a single weekly move as a durable trend without supporting
  year-over-year evidence.
- Explain technical relationships in language a financially informed
  non-specialist can understand.
- Use terms such as "suggests," "is consistent with," and "may indicate" when
  the evidence is not conclusive.
- Distinguish Federal Reserve asset holdings from immediately usable banking
  system liquidity; reserve balances, the Treasury General Account, and ON RRP
  should be interpreted separately.
- Treat a falling Treasury General Account as generally adding reserves to the
  banking system and a rising Treasury General Account as generally draining them.
- Treat ON RRP as a remaining liquidity buffer; if it is close to zero, explain
  that there is little excess ON RRP liquidity left to migrate into markets.
- Assess SOFR, EFFR, and IORB primarily through their spreads and stability,
  rather than treating small daily changes as inherently important.
- Ignore fields that are null.
- Do not use headers, bullet points, a preamble, or a concluding disclaimer.
- Do not repeat the same figure in multiple sentences.
- Convert Fed balance-sheet values from millions of dollars into readable
  billions or trillions.
- Convert bank balance-sheet values from billions of dollars into readable
  billions or trillions.
- Round dollar values sensibly, normally to one decimal place, and rates or
  spreads to no more than one decimal place.

Data:
{json.dumps(data, indent=2, default=str)}
"""
 
client = mlflow.deployments.get_deploy_client("databricks")
 
try:
    response = client.predict(
        endpoint=ENDPOINT,
        inputs={"messages": [{"role": "user", "content": prompt}]},
    )
    briefing_text = response["choices"][0]["message"]["content"][1]["text"]
    print(briefing_text)
except Exception as e:
    raise RuntimeError(f"LLM endpoint call failed for as_of={as_of}: {e}") from e
 
if not briefing_text:
    raise ValueError(f"LLM returned an empty briefing for as_of={as_of}.")

The Federal Reserve’s balance sheet sits at roughly $6.7 trillion, up about $7.4 billion from the prior week and $83.8 billion higher than a year ago, while its reserve balances are around $3.1 trillion, showing a modest weekly increase of $43.8 billion but a year‑over‑year decline of $256.1 billion. These movements suggest that, despite a small recent uptick in reserves, the overall trajectory is a withdrawal of excess liquidity because the long‑term fall in reserves and the rapid rise in the Treasury’s General Account ($0.8 trillion, up $460 billion YoY) are draining cash from the banking system. Commercial banks now hold about $19.6 trillion of total credit and $19.4 trillion of deposits, with deposits falling $74 billion week‑over‑week even as they are $1.1 trillion higher than a year ago. The data imply that credit is expanding aggressively (up $1.1 trillion YoY) and is being funded largely by growing deposits, yet cash assets have slipped $21.9 billion and the weekly deposit dip 

In [0]:
# write response: text > df > gold 

briefing_df = (
    spark.createDataFrame([(as_of, briefing_text)], schema="as_of date, briefing string")
    .withColumn("generated_at", F.current_timestamp())
)
 
MAX_ATTEMPTS = 2
for attempt in range(1, MAX_ATTEMPTS + 1):
    try:
        (
            briefing_df.write
            .format("delta")
            .mode("overwrite")
            .option("replaceWhere", f"as_of = '{as_of.isoformat()}'")
            .saveAsTable(f"{CATALOG}.gold.weekly_briefing")
        )
        break
    except Exception as e:
        if attempt == MAX_ATTEMPTS:
            raise RuntimeError(f"Write failed after {MAX_ATTEMPTS} attempts: {e}") from e
        time.sleep(5)
 
print(f"Briefing for {as_of.isoformat()}:\n{briefing_text}")

Briefing for 2026-07-15:
The Federal Reserve’s balance sheet sits at roughly $6.7 trillion, up about $7.4 billion from the prior week and $83.8 billion higher than a year ago, while its reserve balances are around $3.1 trillion, showing a modest weekly increase of $43.8 billion but a year‑over‑year decline of $256.1 billion. These movements suggest that, despite a small recent uptick in reserves, the overall trajectory is a withdrawal of excess liquidity because the long‑term fall in reserves and the rapid rise in the Treasury’s General Account ($0.8 trillion, up $460 billion YoY) are draining cash from the banking system. Commercial banks now hold about $19.6 trillion of total credit and $19.4 trillion of deposits, with deposits falling $74 billion week‑over‑week even as they are $1.1 trillion higher than a year ago. The data imply that credit is expanding aggressively (up $1.1 trillion YoY) and is being funded largely by growing deposits, yet cash assets have slipped $21.9 billion an